# DTAT391. deeptrack.backend.core

<a href="https://colab.research.google.com/github/DeepTrackAI/DeepTrack2/blob/develop/tutorials/3-advanced-topics/DTAT391_backend.core.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# !pip install deeptrack  # Uncomment if running on Colab/Kaggle.

This advanced tutorial introduces the module [deeptrack.backend.core](../../deeptrack/backend/core.py).

## 1. What is `core.py`?

The `core.py` module is DeepTrack2’s foundation for data management and computational graph construction.

It provides the fundamental classes and abstractions that underpin all DeepTrack2 pipelines, enabling flexible, efficient, and traceable computation.

The key roles of `core.py` are:

- **Data Object Abstractions:**
    Defines simple and validated data containers (`DeepTrackDataObject` and `DeepTrackDataDict`) that store, index, and validate arbitrary data. These classes enable hierarchical and multidimensional organization of complex datasets.

- **Computation Graph Nodes:**
    Implements the `DeepTrackNode` class, which represents a node in a computational graph. Each node can compute, store, and cache values, and can express dependencies on other nodes—enabling the creation of highly flexible and efficient processing pipelines.

- **Lazy Evaluation & Caching:**
    Supports on-demand computation and result caching through lazy evaluation. Nodes only compute their value when required, and cache results for future use until dependencies are invalidated.

- **Operator Overloading for Pipelines:**
    Enables intuitive construction of complex computational graphs using standard Python arithmetic and comparison operators (e.g., +, *, <). This makes pipeline composition both expressive and readable.

- **Dependency Tracking & Propagation:**
    Tracks parent-child and dependency relationships among nodes, so that changes or invalidations automatically propagate through the graph—guaranteeing computational consistency.

- **Citation and Provenance:**
    Integrates citation metadata, ensuring proper academic attribution for work that builds upon DeepTrack2’s infrastructure.

## 2. Using Nodes with Parent-Child Dependencies

In DeepTrack2, nodes represent computational units that can be flexibly linked into graphs by defining dependencies. This allows you to build modular, traceable, and efficient pipelines where changes automatically propagate through the graph.

Below you will see how to set up parent-child relationships between nodes, store and compute data, and propagate invalidation when the upstream data changes.

### 2.1. Creating the Parent and Child Nodes

Create a parent node and a child node whose value is twice that of its parent.

In [2]:
from deeptrack.backend.core import DeepTrackNode

# Create parent and child nodes
parent = DeepTrackNode(action=lambda: 10)
child = DeepTrackNode(action=lambda _ID=None: parent(_ID) * 2)

### 2.2. Establishing Parent-Child Dependency

Link the parent and child so that the child automatically tracks changes in the parent. In this way, this relationship ensures that the child is also kept up to date whenever the parent is updated or invalidated.

In [3]:
# Establish parent-child dependency
parent.add_child(child)

DeepTrackNode(len=0, action=<lambda>)

### 2.3. Storing Values and Computing Results

You can assign different values to the parent at once by associating them with different data indices (`_ID`s).

In [4]:
# Store values in parent node associated to different _IDs
parent.store(15, _ID=(0,))
parent.store(20, _ID=(1,))

DeepTrackNode(len=2, action=<lambda>, IDs=[(0,), (1,)])

### 2.4. Computing and Accessing Child Values

The child node computes its value based on the current value of the parent for each index.

In [5]:
child(_ID=(0,))

30

In [6]:
child(_ID=(1,))

40

**NOTE:** Calling `child(_ID=(0,))` computes the value if needed, and caches it.
On the other hand, calling `child.current_value((0,))` retrieves the currently cached value without recomputing.
Therefore, you can access the last computed value for a specific index using `.current_value(_ID)`.
If the value hasn’t yet been computed or stored, this will raise a `KeyError`.

In [7]:
# Retrieve the cached value without computing
child.current_value((0,))  # Raise KeyError if value not already computed

30

### 2.5. Validating and Invalidating

When you invalidate the parent for a particular `_ID`, the child’s value for that `_ID` will also be marked as invalid (since it depends on the parent). This ensures that downstream computations are never out of sync.

In [8]:
# Invalidate parent data for a given ID
parent.invalidate((0,))
parent.is_valid((0,))

False

In [9]:
child.is_valid((0,))

False

### 2.6. Updating and Recomputing Values

After invalidation, if you update the parent and request the child’s value again, it will be recomputed.

In [10]:
# Update the parent value and recompute the child value
parent.store(25, _ID=(0,))
child((0,))

50

In [11]:
parent.is_valid((0,))

True

In [12]:
child.is_valid((0,))

True

### 2.7 Setting a Value and Automatic Invalidation

You can force a value into a node’s storage with `.set_value(value, _ID)`. If the new value is different, dependent nodes will be invalidated.

In [13]:
parent.set_value(100, _ID=(1,))
parent.current_value((1,))

100

In [14]:
parent.is_valid((1,))

True

In [15]:
child.is_valid((1,))

False

### 2.8. Evaluating, Resetting, and Recomputing Node Values

The value of a `DeepTrackNode` is evaluated and stored when the node is called with `.__call__()` (e.g., `node()`). When you call a node multiple times, you will always get the same value as the node value is retrieved from memory (and not recomputed) each time.

The state of the node (and its dependencies) can be reset using the `.update()` method. After this, calling the node will result in a recomputation of its value (e.g., `node.update()()`).

For convenience, you can use the `.new()` method instead of `.update()()`.

#### 2.8.1. Calling a Node

You can evaluate a node by calling it with a specific (optional) `_ID`. This triggers the `__call__()` method:

In [16]:
parent = DeepTrackNode(lambda: 10)
child = DeepTrackNode(lambda _ID=None: parent(_ID) * 2)
parent.add_child(child)

child((0,))  # Triggers computation for both parent and child

20

If the node already has valid stored data for the given `_ID`, it returns that directly (cached result). Otherwise, it computes the value using the action, stores it, and then returns it. Thus, repeated calls with the same _ID will return cached results, unless invalidated

#### 2.8.2. Resetting a Node

In most cases, a `DeepTrackNode` automatically caches and reuses previously computed values unless explicitly invalidated. However, sometimes you may want to force a fresh computation; for instance, when the node's output is stochastic or time-dependent.

The `.update()` method is provided exactly for this purpose: It clears all stored data in a node and its entire dependency graph. This invalidates everything and removes all cached results.

In [17]:
parent.store(20, _ID=(0,))
print(parent((0,)))  # Output: 20 (from cache)

20


After calling `.update()`, the next evaluation will recompute all values.

In [18]:
parent.update()
print(parent((0,)))  # Output: 10 (recomputed from action)

10


#### 2.8.3. Using `.new()` to Recompute a Node's Value

The `.new()` method is equivalent to using `.update()()`, i.e., the `.update()` method to clear the values sotred in the node and its dependencies followed by the `.__call__()` method to recompute the values from scratch.

For example, consider a node returning a random value.

In [19]:
import random

node = DeepTrackNode(lambda _ID=None: random.randint(0, 100))

Calling

In [20]:
node.new()

32

is equivalent to (but more elegant than)

In [21]:
node.update()()

77

### 2.9. Creating and Visualizing a Complex Graph

Now create a complex computational graph with the following structure:

        ┌────────┐         ┌────────┐
        │ input1 │         │ input2 │
        └────────┘         └────────┘
            ↓                  ↓
      ┌────────────┐     ┌────────────┐
      │ process1_A │     │ process2_A │
      └────────────┘     └────────────┘
            ↓                  ↓
      ┌────────────┐     ┌────────────┐
      │ process1_B │     │ process2_B │
      └────────────┘     └────────────┘
                    ↘  ↙
               ┌────────────┐
               │  merger    │
               └────────────┘
                    ↓
           ┌────────────────┐
           │  postprocess   │
           └────────────────┘
               ↓       ↓
          ┌──────┐   ┌──────┐
          │ out1 │   │ out2 │
          └──────┘   └──────┘

In [22]:
# Shared input nodes
input1 = DeepTrackNode(lambda: 3, node_name="input1")
input2 = DeepTrackNode(lambda: 5, node_name="input2")

# Independent preprocessing
process1_A = DeepTrackNode(lambda _ID=None: input1(_ID) + 1,
                           node_name="process1_A")
process2_A = DeepTrackNode(lambda _ID=None: input2(_ID) * 2,
                           node_name="process2_A")

process1_B = DeepTrackNode(lambda _ID=None: process1_A(_ID) ** 2,
                           node_name="process1_B")
process2_B = DeepTrackNode(lambda _ID=None: process2_A(_ID) - 1,
                           node_name="process2_B")

# Merge branch: sum both processed paths
merger = DeepTrackNode(
    lambda _ID=None: process1_B(_ID) + process2_B(_ID),
    node_name="merger"
)

# Post-processing
postprocess = DeepTrackNode(lambda _ID=None: merger(_ID) / 2,
                            node_name="postprocess")

# Split again
out1 = DeepTrackNode(lambda _ID=None: postprocess(_ID) + 100, node_name="out1")
out2 = DeepTrackNode(lambda _ID=None: postprocess(_ID) * 3, node_name="out2")

# Link nodes
input1.add_child(process1_A)
input2.add_child(process2_A)

process1_A.add_child(process1_B)
process2_A.add_child(process2_B)

process1_B.add_child(merger)
process2_B.add_child(merger)

merger.add_child(postprocess)

postprocess.add_child(out1)
postprocess.add_child(out2);

You can print the children of, for example, `input1`.

In [23]:
input1.print_children_tree()

- DeepTrackNode 'input1' at 0x337a34e50
    - DeepTrackNode 'process1_A' at 0x337a34e80
        - DeepTrackNode 'process1_B' at 0x337a351b0
            - DeepTrackNode 'merger' at 0x337a35690
                - DeepTrackNode 'postprocess' at 0x337a34160
                    - DeepTrackNode 'out1' at 0x337a367a0
                    - DeepTrackNode 'out2' at 0x337a37730


You can also print the dependencies of, for example, `out2`.

In [24]:
out2.print_dependencies_tree()

- DeepTrackNode 'out2' at 0x337a37730
    - DeepTrackNode 'postprocess' at 0x337a34160
        - DeepTrackNode 'merger' at 0x337a35690
            - DeepTrackNode 'process1_B' at 0x337a351b0
                - DeepTrackNode 'process1_A' at 0x337a34e80
                    - DeepTrackNode 'input1' at 0x337a34e50
            - DeepTrackNode 'process2_B' at 0x337a35120
                - DeepTrackNode 'process2_A' at 0x337a34f10
                    - DeepTrackNode 'input2' at 0x337a34df0


## 3. Performing Lazy Evaluation and Caching

A powerful feature of DeepTrack2 nodes is lazy evaluation: the node’s value is only computed when it is needed, and the result is cached until the node (or its dependencies) is invalidated. This avoids redundant computations and ensures high efficiency, especially in large graphs.

In this example, you will use a global counter to demonstrate when the node’s computation actually happens.

### 3.1 Defining a Node with a Side Effect

First, define a calculation function that increments a global counter each time it is called. This will allow you to see exactly how many times the node’s computation is performed.

In [25]:
# Create counter node with side effect
call_count = 0
def calculation():
    global call_count
    call_count += 1
    return 10

node = DeepTrackNode(calculation)

### 3.2 Demonstrating Lazy Evaluation

Let’s see what happens when we call the node multiple times:

In [26]:
# First call computes the value (calls the function)
node()

10

In [27]:
call_count

1

In [28]:
# Second call uses the cached value (no additional computation)
node()

10

In [29]:
call_count

1

### 3.3 Forcing Recalculation though Invalidation

If you invalidate the node, the cache is cleared and the next call will recompute the value:

In [30]:
# Invalidate the node and call again (forces recomputation)
node.invalidate()
node()

10

In [31]:
call_count

2

In [32]:
# Invalidate and call again
node.invalidate()
node()

10

In [33]:
call_count

3

You would obtain some similar results using the `.update()` or `.new()` methods.

## 4. Managing Data with IDs

The `DeepTrackDataDict` class provides an efficient, validated way to manage multiple data objects, each indexed by a unique tuple of integers.

**NOTE:** This is particularly relevant when using the `Repeat` feature (accessed also throught the`^` operator).

### 4.1. Creating and Indexing Data Objects

You can create entries with arbitrary integer index tuples, just like keys in a dictionary.

In [34]:
from deeptrack.backend.core import DeepTrackDataDict

data_dict = DeepTrackDataDict()

# Create listings with unique indices.
data_dict.create_index((0, 0))
data_dict.create_index((0, 1))
data_dict.create_index((1, 0))
data_dict.create_index((1, 1))

### 4.2. Storing and Retrieving Data

Each index corresponds to a `DeepTrackDataObject`, where you can store and retrieve data.
This is similar to using a multidimensional dictionary.

In [35]:
# Store some data for the indices.
data_dict[(0, 0)].store("Cat")
data_dict[(0, 1)].store("Dog")
data_dict[(1, 0)].store("Mouse")
data_dict[(1, 1)].store("Bird")

### 4.3. Accessing Data by ID

You can access data by its full index, or get a dictionary of all entries with a common prefix.

In [36]:
# Retrieve and print values for specific indices.
data_dict[(0, 0)].current_value()

'Cat'

In [37]:
data_dict[(1, 1)].current_value()

'Bird'

In [38]:
# Retrieve all entries whose indices start with (0,)
data_dict[(0, )]

{(0, 0): DeepTrackDataObject(data='Cat', valid=True),
 (0, 1): DeepTrackDataObject(data='Dog', valid=True)}

## 5. Overloading Operators and Composing Pipeline

A unique and powerful feature of `DeepTrackNode` is its support for operator overloading. This allows you to build complex computational pipelines by composing nodes using familiar arithmetic and comparison operators, making your code both expressive and readable.

Every operator creates a new node that, when called, evaluates its operands, applies the operator, and caches the result. Dependency relationships are automatically tracked, so invalidating an operand will invalidate any composed nodes as well.

### 5.1. Combining Nodes with Arithmetic Operators

You can add, subtract, multiply, or divide nodes just like numbers. The result is always a new `DeepTrackNode` that represents the composed computation.

In [39]:
from deeptrack.backend.core import DeepTrackNode

a = DeepTrackNode(lambda: 5)
b = DeepTrackNode(lambda: 3)

In [40]:
sum_node = a + b
sum_node()

8

In [41]:
diff_node = a - b
diff_node()

2

In [42]:
prod_node = a * 2
prod_node()

10

In [43]:
div_node = a / b
div_node()

1.6666666666666667

In [44]:
floordiv_node = a // b
floordiv_node()

1

### 5.2. Chaining and Nesting Operators

You can compose pipelines of arbitrary depth and complexity.

In [45]:
complex_node = ((a + b) * 2) / (b + 1)
complex_node()

4.0

### 5.3. Using Comparison Operators for Graphs

Comparison operators also work on nodes, returning new nodes that compute boolean results.

In [46]:
lt_node = a < b
lt_node()

False

In [47]:
ge_node = a >= b
ge_node()

True

### 5.4. Mixing Nodes and Constants

You can mix DeepTrackNode instances and regular numbers.

In [48]:
sum_with_constant = a + 7
sum_with_constant()

12

In [49]:
mult_with_constant = 3 * b
mult_with_constant()

9

## 6. Getting Citations

The `DeepTrackNode` class can also be used to obtain the relevant citations.

In [50]:
DeepTrackNode().get_citations()

{'\n@article{Midtvet2021Quantitative,\n    author  = {Midtvedt, Benjamin and Helgadottir, Saga and Argun, Aykut and \n               Pineda, Jesús and Midtvedt, Daniel and Volpe, Giovanni},\n    title   = {Quantitative digital microscopy with deep learning},\n    journal = {Applied Physics Reviews},\n    volume  = {8},\n    number  = {1},\n    pages   = {011310},\n    year    = {2021},\n    doi     = {10.1063/5.0034891}\n}\n'}